<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

# 학습 내용
>이번 장에서는 <strong>Reflection 패턴(Reflection Pattern)</strong>에 대해 학습합니다.
>Single Agent의 한계를 체감하고, Supervisor가 여러 Worker를 관리하는 Fan-out/Fan-in 멀티 에이전트 패턴과 성과 평가 함수를 학습해봅시다.

# Single Agent 한계 체감
> 복잡한 작업을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">단일 Agent</mark>에게 맡기면 어떤 문제가 발생하는지 직접 확인합니다.

Multi-Agent의 필요성을 이해하려면 먼저 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">단일 Agent의 한계</mark>를 직접 체감해야 합니다. 하나의 LLM이 계획 수립, 실행, 검토를 모두 처리하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정보 누락</mark>, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">깊이 부족</mark>, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">체계성 저하</mark>가 발생할 수 있습니다.

<div style="text-align:center">

</div>

In [ ]:
# TODO 0: 단일 Agent로 복잡한 작업을 시도하고 한계를 확인해봅시다. llm.invoke()로 복잡한 요청을 처리하세요.

complex_request = """
다음 작업을 한 번에 수행해주세요:
1. AI 시장 동향을 3가지 관점(기술, 투자, 규제)에서 분석
2. 각 관점별 핵심 수치와 출처를 포함
3. 3개월 후 전망을 구체적 근거와 함께 제시
4. 분석 결과를 표 형태로 정리
5. 전체 분석의 신뢰도를 자체 평가
"""

single_agent_result = llm.invoke(complex_request)
print("=== Single Agent 응답 ===")
print(single_agent_result.content[:600])
print("..." if len(single_agent_result.content) > 600 else "")
print(f"\n응답 길이: {len(single_agent_result.content)}자")
print("\n⚠️ 확인해보세요:")
print("  - 5가지 요청이 모두 충족되었나요?")
print("  - 수치와 출처가 구체적인가요?")
print("  - 자체 평가가 객관적인가요?")

# Supervisor 패턴 (Supervisor Pattern)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">중앙 관리자(Supervisor)</mark>가 여러 Worker에게 작업을 분배하고 결과를 종합합니다.

LLM이 첫 번째 시도에서 최적의 결과를 내는 경우는 드뭅니다. 사람도 글을 쓸 때 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">초안 작성 → 검토 → 수정</mark>의 과정을 반복하듯, AI도 동일한 프로세스를 따를 때 품질이 크게 향상됩니다. <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">첫 결과의 불완전성</mark>으로 초기 응답에는 누락된 내용이나 논리적 허점이 있을 수 있고, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">반복적 개선</mark>을 통해 Supervisor가 Worker의 결과를 평가하고 재작업을 지시하면 품질이 점진적으로 향상됩니다. 또한 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">자동화된 품질 관리</mark>로 사람이 매번 검토하지 않아도 Agent 스스로 기준에 미달한 결과를 걸러낼 수 있습니다. 이 패턴은 보고서 작성, 코드 생성, 콘텐츠 제작 등 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">품질 기준이 명확한 반복 작업</mark>에 특히 효과적입니다.</br>
이 내용을 학습하기 전에 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Agent</mark>(자율 실행 구조와 상태 관리, Ch.4-2-1_001 참고), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">StateGraph</mark>(조건부 엣지와 루프 구현 방법, Ch.4-2-1_002 참고), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Planner-Worker 패턴</mark>(계획-실행 분리 구조와 <code>AgentState</code> 설계, Ch.4-2-2_001 참고)의 개념을 먼저 이해하면 좋습니다.

## SupervisorState 정의
> Annotated + operator.add로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">여러 Worker의 결과를 자동 누적</mark>합니다.

In [ ]:
# TODO 1: 상태 타입을 정의하세요. 필드는 task(str), worker_results(리스트 누적 방식), next_action(str), final_output(str)입니다. 리스트 결합 동작을 확인하세요.

from typing import TypedDict, List, Annotated
import operator

class SupervisorState(TypedDict):
    task: str
    worker_results: Annotated[List[str], operator.add]
    next_action: str
    final_output: str

# operator.add 동작 확인
existing = ["결과1"]
new = ["결과2"]
combined = operator.add(existing, new)
print(f"기존: {existing}")
print(f"추가: {new}")
print(f"결합: {combined}")

## Fan-out / Fan-in 패턴
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Fan-out</mark>: 하나의 노드에서 여러 노드로 작업 분배
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Fan-in</mark>: 여러 노드의 결과를 하나로 모으기

In [ ]:
# TODO 2: supervisor(결과 없으면 "distribute", 있으면 "synthesize" 반환), worker_research("[조사]" 접두어로 결과 반환), worker_analysis("[분석]" 접두어로 결과 반환), synthesize(worker 결과를 종합하여 최종 출력 반환) 4개의 노드 함수를 정의하세요.

def supervisor(state: SupervisorState) -> SupervisorState:
    """작업 분배 및 다음 행동 결정"""
    if not state.get("worker_results"):
        return {"next_action": "distribute"}
    else:
        return {"next_action": "synthesize"}

def worker_research(state: SupervisorState) -> SupervisorState:
    """조사 담당 Worker"""
    result = llm.invoke(f"조사: {state['task']}").content
    return {"worker_results": [f"[조사] {result[:50]}..."]}

def worker_analysis(state: SupervisorState) -> SupervisorState:
    """분석 담당 Worker"""
    result = llm.invoke(f"분석: {state['task']}").content
    return {"worker_results": [f"[분석] {result[:50]}..."]}

def synthesize(state: SupervisorState) -> SupervisorState:
    """결과 종합"""
    combined = "\n".join(state["worker_results"])
    final = llm.invoke(f"종합하세요:\n{combined}").content
    return {"final_output": final}

## 그래프 조립

In [ ]:
# TODO 3: 라우팅 분기 함수를 정의하고, 상태 그래프를 조립하세요. 흐름은 START→supervisor→(distribute: research→analysis→supervisor, synthesize: synthesize→종료)입니다. "AI 시장 동향 분석"으로 실행하여 결과를 출력하세요.

def route_supervisor(state: SupervisorState) -> str:
    return state["next_action"]

graph = StateGraph(SupervisorState)
graph.add_node("supervisor", supervisor)
graph.add_node("research", worker_research)
graph.add_node("analysis", worker_analysis)
graph.add_node("synthesize", synthesize)

graph.add_edge(START, "supervisor")
graph.add_conditional_edges("supervisor", route_supervisor, {
    "distribute": "research",  # Fan-out
    "synthesize": "synthesize"
})
graph.add_edge("research", "analysis")
graph.add_edge("analysis", "supervisor")  # Fan-in
graph.add_edge("synthesize", END)

app = graph.compile()
result = app.invoke({"task": "AI 시장 동향 분석", "worker_results": []})
print(f"Worker 결과 수: {len(result['worker_results'])}")
for r in result["worker_results"]:
    print(f"  {r[:60]}...")
print(f"\n최종 출력: {result['final_output'][:100]}...")

## 성과 평가 함수 (Performance Evaluation)
> Multi-Agent 결과의 품질을 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LLM 기반 점수 산출</mark>로 정량적으로 측정합니다.

Multi-Agent 시스템이 잘 작동하는지 확인하려면 결과를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정량적으로 평가</mark>해야 합니다. 단순히 "좋아 보인다"가 아니라, <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Planner 계획의 완성도</mark>(30점), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Worker 실행의 충실도</mark>(40점), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">전체 요청 충족도</mark>(30점)를 기준으로 100점 만점의 점수를 산출합니다. LLM에게 평가 기준과 Agent 상태를 제공하고, JSON 형식으로 점수와 피드백을 받아 파싱합니다.

<div style="text-align:center">

</div>

In [ ]:
# TODO 4: evaluate_multi_agent() 성과 평가 함수를 구현하세요. LLM에게 평가 기준(Planner 30점, Worker 40점, 전체 30점)과 상태를 제공하고, JSON으로 점수와 피드백을 파싱하여 반환하세요.

import json
import re

def evaluate_multi_agent(state: SupervisorState) -> dict:
    """
    Multi-Agent 시스템의 성과를 정량적으로 평가합니다.

    Args:
        state: 현재 SupervisorState (task, worker_results, final_output 포함)

    Returns:
        dict: {"score": int, "feedback": str, "details": dict}
    """
    # TODO 4-1: 평가 프롬프트를 구성하세요.
    eval_prompt = f"""당신은 Multi-Agent 시스템의 성과를 평가하는 평가자입니다.

## 평가 기준
1. Planner 평가 (0~30점): 작업 분배의 적절성, Worker 역할 구분의 명확성
2. Worker 평가 (0~40점): 실행 품질, 정보의 충실도와 정확성
3. 전체 평가 (0~30점): 사용자 요청 충족도, 최종 결과물의 완성도

## 입력 정보
- 사용자 요청: {state['task']}
- Worker 결과들: {state.get('worker_results', [])}
- 최종 출력: {state.get('final_output', '없음')}

## 출력 형식
반드시 아래 JSON 형식만 출력하세요. 다른 텍스트는 포함하지 마세요.
{{"planner_score": <0~30>, "worker_score": <0~40>, "overall_score": <0~30>, "feedback": "<전체 피드백>"}}"""

    # TODO 4-2: LLM을 호출하여 평가를 수행하세요.
    response = llm.invoke(eval_prompt)
    content = response.content

    # TODO 4-3: JSON을 파싱하세요. 마크다운 코드 블록(```json ... ```) 처리를 포함합니다.
    try:
        json_match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', content)
        if json_match:
            json_str = json_match.group(1)
        else:
            json_match = re.search(r'\{[\s\S]*\}', content)
            json_str = json_match.group(0) if json_match else content

        parsed = json.loads(json_str)
        total_score = parsed['planner_score'] + parsed['worker_score'] + parsed['overall_score']
        return {
            "score": total_score,
            "feedback": parsed['feedback'],
            "details": {
                "planner": parsed['planner_score'],
                "worker": parsed['worker_score'],
                "overall": parsed['overall_score']
            }
        }
    except Exception as e:
        return {"score": 0, "feedback": f"평가 실패: {str(e)}\n원본 응답: {content[:200]}", "details": {}}

# 평가 실행 (이전 그래프 실행 결과 사용)
evaluation = evaluate_multi_agent(result)
print("=== 성과 평가 결과 ===")
print(f"종합 점수: {evaluation['score']}/100")
if evaluation['details']:
    print(f"세부 점수: Planner {evaluation['details']['planner']}/30, "
          f"Worker {evaluation['details']['worker']}/40, "
          f"전체 {evaluation['details']['overall']}/30")
print(f"피드백: {evaluation['feedback']}")

💡성과 평가 함수의 핵심
> LLM에게 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">명확한 평가 기준</mark>과 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">JSON 출력 형식</mark>을 지정하면 정량적 평가가 가능합니다.
> `re.search()`로 마크다운 코드 블록 안의 JSON을 추출하는 패턴은 LLM 응답 파싱에서 자주 사용됩니다.
> 평가 항목을 세분화(Planner/Worker/Overall)하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">어느 Agent에서 병목</mark>이 발생하는지 진단할 수 있습니다.

## 멀티 에이전트 패턴 비교

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">패턴</th>
      <th>구조</th>
      <th>특징</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">Planner-Worker</td><td>순차 (계획→실행)</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">단일 Worker</mark>, 반복 개선</td></tr>
    <tr><td style="text-align:center">Supervisor</td><td>Fan-out/Fan-in</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">다수 Worker</mark>, 병렬 처리</td></tr>
    <tr><td style="text-align:center">Reflection</td><td>자기 평가 루프</td><td>결과 품질 자동 검증</td></tr>
  </tbody>
</table>

💡Annotated[List, operator.add]의 핵심
> 일반 딕셔너리 업데이트는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">값을 덮어씁니다</mark>.
> `operator.add`를 사용하면 리스트에 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">추가(append)</mark>됩니다.
> 이를 통해 여러 Worker의 결과를 자동으로 누적할 수 있습니다.

💡언제 어떤 패턴을 쓸까?
> 단순 작업: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Direct 패턴</mark> (1회 호출)
> 복잡한 단일 작업: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ReAct 패턴</mark> (반복 추론)
> 작업 분해 필요: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Planner-Worker</mark> (계획 + 실행)
> 병렬 전문가 필요: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Supervisor 패턴</mark> (Fan-out/Fan-in)